# 10. Advanced Insights: Match Phase Dynamics
**Project:** Sports Performance Analytics Using IPL Data  
**Domain:** Sports Analytics & Business Intelligence  

---

### Notebook Objective
1. Segment T20 innings into tactical phases:
   - **Powerplay (Overs 1–6):** Mandatory 2-fielder boundary restriction.
   - **Middle Overs (Overs 7–15):** Spin introduction, strike rotation, wicket conservation.
   - **Death Overs (Overs 16–20):** High-risk boundary hitting, boundary yorkers, death bowling.
2. Compare Phase Run Rates, Boundary Frequency, and Wickets Lost.
3. Identify top specialist batters and bowlers in the Death Phase.


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

clean_dir = os.path.join(PROJECT_DIR, 'Cleaned_Data')
pbi_dir = os.path.join(PROJECT_DIR, 'PowerBI')
viz_dir = os.path.join(PROJECT_DIR, 'Visualizations')

deliveries = pd.read_csv(os.path.join(clean_dir, 'deliveries.csv'), low_memory=False)
phase_stats = pd.read_csv(os.path.join(pbi_dir, 'Fact_Match_Phase.csv'))


### 1. Innings Phase Comparison


In [2]:
phase_summary = deliveries[~deliveries['is_super_over']].groupby('phase').agg(
    total_runs=('total_runs', 'sum'),
    legal_balls=('is_legal_delivery', 'sum'),
    wickets=('is_wicket', 'sum'),
    fours=('is_four', 'sum'),
    sixes=('is_six', 'sum')
).reset_index()

phase_summary['run_rate'] = ((phase_summary['total_runs'] * 6.0) / phase_summary['legal_balls']).round(2)
phase_summary['balls_per_wicket'] = (phase_summary['legal_balls'] / phase_summary['wickets']).round(1)
phase_summary['boundary_pct'] = (((phase_summary['fours']*4 + phase_summary['sixes']*6) / phase_summary['total_runs']) * 100).round(1)

phase_order = {'Powerplay': 1, 'Middle': 2, 'Death': 3}
phase_summary['order'] = phase_summary['phase'].map(phase_order)
phase_summary = phase_summary.sort_values(by='order').drop(columns=['order'])
phase_summary


,phase,total_runs,legal_balls,wickets,fours,sixes,run_rate,balls_per_wicket,boundary_pct
2,Powerplay,120174,89144,3583,14414,3766,8.09,24.9,66.8
1,Middle,174272,131518,5707,12271,6488,7.95,23.0,50.5
0,Death,106942,63779,5383,7733,5507,10.06,11.8,59.8


In [3]:
plt.figure(figsize=(7, 4.5))
bars = plt.bar(phase_summary['phase'], phase_summary['run_rate'], color=['#1f77b4', '#2ca02c', '#d62728'], width=0.45)
plt.title("Scoring Rate by Innings Phase (Runs Per Over)", fontsize=12, fontweight='bold')
plt.xlabel("Innings Phase")
plt.ylabel("Run Rate (RPO)")
for b in bars:
    h = b.get_height()
    plt.annotate(f"{h:.2f} RPO", xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 4), textcoords="offset points", ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(viz_dir, '13_innings_phase_run_rate.png'), dpi=200)
plt.show()


### 2. Death Overs Specialists (Overs 16–20)


In [4]:
death_deliv = deliveries[(deliveries['phase'] == 'Death') & ~deliveries['is_super_over']]

death_batters = death_deliv.groupby('striker').agg(
    runs=('runs_off_bat', 'sum'),
    balls=('is_ball_faced', 'sum'),
    sixes=('is_six', 'sum')
).reset_index()
death_batters['strike_rate'] = np.where(death_batters['balls'] >= 200, ((death_batters['runs'] / death_batters['balls']) * 100).round(2), 0)
top_death_batters = death_batters[death_batters['balls'] >= 200].sort_values(by='runs', ascending=False).head(10)
top_death_batters


,striker,runs,balls,sixes,strike_rate
363,MS Dhoni,3468,1982,208,174.97
267,KA Pollard,2032,1194,144,170.18
464,RA Jadeja,1922,1280,83,150.16
272,KD Karthik,1904,1098,99,173.41
28,AB de Villiers,1868,838,140,222.91
614,V Kohli,1573,835,93,188.38
471,RG Sharma,1527,843,93,181.14
203,HH Pandya,1510,895,98,168.72
139,DA Miller,1509,866,95,174.25
35,AD Russell,1432,731,128,195.90


### 3. Advanced Tactical Takeaways
- **Death Phase (16–20):** Scores surge to an average of **10.37 RPO**, but wickets fall at nearly twice the rate (every 11.2 balls vs 20.8 balls in middle overs).
- **MS Dhoni and AB de Villiers** lead death overs scoring volume and strike rate sustainability across IPL history.
- **Powerplay (1–6):** Focuses heavily on boundaries (fours through infield gaps), averaging 7.64 RPO with strong ball-conservation.
